# VGG19 ImageNet inference

This notebook follows the official torchvision pretrained-model pattern: choose a pretrained weights object, use its matching transforms, run inference, and decode the top predictions. The goal is to understand the full VGG19 image-classification path before reusing VGG19 feature layers in later labs.

In [31]:
from torchvision.models import VGG19_Weights, vgg19

# Choose the official pretrained VGG19 ImageNet weights.
# In torchvision, the weights object also carries the matching preprocessing recipe and labels.
weights = VGG19_Weights.DEFAULT

# Build VGG19 with the pretrained weights and switch to inference/evaluation mode.
model = vgg19(weights=weights)
model.eval()

# Get the input recipe and output label names that match these exact weights.
preprocess = weights.transforms()
categories = weights.meta["categories"]

preprocess, categories[:10]

(ImageClassification(
     crop_size=[224]
     resize_size=[256]
     mean=[0.485, 0.456, 0.406]
     std=[0.229, 0.224, 0.225]
     interpolation=InterpolationMode.BILINEAR
 ),
 ['tench',
  'goldfish',
  'great white shark',
  'tiger shark',
  'hammerhead',
  'electric ray',
  'stingray',
  'cock',
  'hen',
  'ostrich'])

## Resolve project paths and choose an image

Notebook paths are relative to the kernel's current working directory, not always to the notebook file. This cell finds the project root and points to one local image under `data/` so later outputs stay organized.

In [32]:
from pathlib import Path

cwd = Path.cwd()
project_root = cwd if (cwd / "pyproject.toml").exists() else cwd.parent

image_path = project_root / "data" / "keyboard.jpg"

cwd, project_root, image_path.exists(), image_path

(WindowsPath('C:/Users/giloz/dev/visual-genai-lab/notebooks'),
 WindowsPath('C:/Users/giloz/dev/visual-genai-lab'),
 True,
 WindowsPath('C:/Users/giloz/dev/visual-genai-lab/data/keyboard.jpg'))

## Preprocess the image

The torchvision transform converts a normal RGB image into the tensor format VGG19 expects. The key shape change is from a display image size like `(width, height)` to a model input tensor shaped `[channels, height, width]`, usually `[3, 224, 224]` for this VGG19 recipe.

In [33]:
from PIL import Image

image = Image.open(image_path).convert("RGB")

# Apply the official VGG19 transform: resize, crop, convert to tensor, and normalize.
image_tensor = preprocess(image)

image.size, image_tensor.shape

((5712, 4284), torch.Size([3, 224, 224]))

## Run inference and decode top-5 predictions

VGG19 expects a batch shaped `[N, C, H, W]`, so one image needs an added batch dimension. The model returns 1000 raw ImageNet scores; `softmax` turns them into probabilities, and `topk(5)` keeps the five strongest predictions. Wrapping these steps in a function lets the same inference path run on more than one image without copying the code.

In [34]:
import torch


def predict_image(image_path: Path, top_k: int = 5) -> list[tuple[str, float]]:
    # Load and preprocess the image the same way every time.
    image = Image.open(image_path).convert("RGB")
    image_tensor = preprocess(image)

    # VGG19 expects a batch shaped [N, C, H, W].
    # unsqueeze(0) wraps this one image tensor (C, H, W) into a batch of size 1.
    batch = image_tensor.unsqueeze(0)

    # inference_mode disables gradient tracking because we are predicting, not training.
    with torch.inference_mode():
        scores = model(batch)

    # Convert this image's 1000 raw ImageNet scores into probabilities.
    probabilities = scores.squeeze(0).softmax(dim=0)

    # Keep the highest-probability class IDs and pair them with readable labels.
    top_probabilities, top_class_ids = probabilities.topk(top_k)

    return [
        (categories[class_id.item()], float(probability))
        for probability, class_id in zip(top_probabilities, top_class_ids, strict=False)
    ]


top5_predictions = predict_image(image_path)
top5_predictions

[('computer keyboard', 0.8361623287200928),
 ('mouse', 0.0713619738817215),
 ('space bar', 0.02965741977095604),
 ('typewriter keyboard', 0.013880019076168537),
 ('remote control', 0.008667639456689358)]

In [35]:
image_paths = [
       project_root / "data" / "keyboard.jpg",
       project_root / "data" / "panda.jpg",
   ]

all_predictions = {
       path.name: predict_image(path)
       for path in image_paths
   }

all_predictions

{'keyboard.jpg': [('computer keyboard', 0.8361623287200928),
  ('mouse', 0.0713619738817215),
  ('space bar', 0.02965741977095604),
  ('typewriter keyboard', 0.013880019076168537),
  ('remote control', 0.008667639456689358)],
 'panda.jpg': [('giant panda', 0.9998574256896973),
  ('badger', 4.550008088699542e-05),
  ('sloth bear', 3.394263330847025e-05),
  ('lesser panda', 3.017786002601497e-05),
  ('American black bear', 2.1529333025682718e-05)]}